In [1]:
!pwd

/Users/shovonsengupta/Desktop/All/Time_Series_Forecasting_Research/multi_variate_forecasting_paper_G7/GitHub_Macrocasting/code/Algorithms


In [2]:
import sys;
print(sys.version)

3.10.6 (main, Aug 10 2024, 10:05:53) [Clang 15.0.0 (clang-1500.3.9.4)]


In [1]:
# !pip install uv

# !uv pip install \
#   "darts==0.34.0" \
#   "torch==2.3.1" \
#   "pytorch-lightning==2.2.5" \
#   "torchmetrics==1.3.2" \
#   "numpy==1.26.4" \
#   "pandas==2.2.2" \
#   "scikit-learn==1.7.2" \
#   "dask==2025.5.0" \
#   "xgboost==3.0.2" \
#   "catboost==1.2.8" \
#   "pyyaml==6.0.2" \
#   "lightgbm==4.6.0"

# Pin torch for reproducibility with your local run
# !uv pip install --index-url https://download.pytorch.org/whl/cu121 "torch==2.3.1"

Using Python 3.12.13 environment at: /usr
Checked 1 package in 90ms


In [1]:
import sys, torch, darts, pytorch_lightning as pl, numpy as np, pandas as pd
print(sys.version)
print("torch:", torch.__version__)
print("darts:", darts.__version__)
print("lightning:", pl.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda:", torch.cuda.is_available())

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda: True


**Forecasts**
 - **Country: Canada**
 - **Forecast Horizons:12M and 24M Forecast**

In [2]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class canadaForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the forecast generator for canada data

      Args:
          data_path: Path to the canada data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> NBEATSModel:
      """
      Create and return the N-BEATS model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured NBEATSModel
      """
      return NBEATSModel(
          input_chunk_length=3,  #forecast_horizon,
          output_chunk_length=forecast_horizon,
          num_stacks=12,
          num_blocks=2,
          num_layers=5,
          layer_widths=16,
          expansion_coefficient_dim=6,
          trend_polynomial_degree=2,
          n_epochs=50,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'canada_forecasts_nbeatsx_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the forecast generation
  """
  # Initialize forecast generator
  generator = canadaForecastGenerator(
      data_path='all_mulvar_data_canada_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"canada_forecasts_nbeatsx_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# canada_forecasts_12m.csv
# canada_forecasts_24m.csv


Generating 12-month forecast...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Training model for 12-month horizon...


INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params
-----------------------------------------------------
0 | criterion       | MSELoss          | 0     
1 | train_criterion | MSELoss          | 0     
2 | val_criterion   | MSELoss          | 0     
3 | train_metrics   | MetricCollection | 0     
4 | val_metrics     | MetricCollection | 0     
5 | stacks          | ModuleList       | 64.4 K
-----------------------------------------------------
64.1 K    Trainable 

Training: |          | 0/? [00:00<?, ?it/s]

...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to canada_forecasts_nbeatsx_24m.csv

Created files:
canada_forecasts_nbeatsx_12m.csv
canada_forecasts_nbeatsx_24m.csv


**Forecasts**
 - **Country: USA**
 - **Forecast Horizons:12M and 24M Forecast**

In [3]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class USAForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the forecast generator for USA data

      Args:
          data_path: Path to the USA data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> NBEATSModel:
      """
      Create and return the N-BEATS model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured NBEATSModel
      """
      return NBEATSModel(
          input_chunk_length=3,  #forecast_horizon,
          output_chunk_length=forecast_horizon,
          num_stacks=12,
          num_blocks=2,
          num_layers=5,
          layer_widths=16,
          expansion_coefficient_dim=6,
          trend_polynomial_degree=2,
          n_epochs=50,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'usa_forecasts_nbeatsx_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the forecast generation
  """
  # Initialize forecast generator
  generator = USAForecastGenerator(
      data_path='all_mulvar_data_usa_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"usa_forecasts_nbeatsx_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# usa_forecasts_12m.csv
# usa_forecasts_24m.csv

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params
-----------------------------------------------------
0 | criterion       | MSELoss          | 0     
1 | train_criterion | MSELoss          | 0     
2 | val_criterion   | MSELoss          | 0     
3 | train_metrics   | MetricCollection | 0     
4 | val_metrics     | MetricCollection | 0     
5 | stacks          | ModuleList       | 64.4 K
-----------------------------------------------------
64.1 K    Trainable params
291       Non-trainable params
64.4 K    Total params
0.258     Total esti


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to usa_forecasts_nbeatsx_24m.csv

Created files:
usa_forecasts_nbeatsx_12m.csv
usa_forecasts_nbeatsx_24m.csv


**Forecasts**
 - **Country: France**
 - **Forecast Horizons:12M and 24M Forecast**

In [4]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class franceForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the forecast generator for france data

      Args:
          data_path: Path to the france data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> NBEATSModel:
      """
      Create and return the N-BEATS model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured NBEATSModel
      """
      return NBEATSModel(
          input_chunk_length=3,  #forecast_horizon,
          output_chunk_length=forecast_horizon,
          num_stacks=12,
          num_blocks=2,
          num_layers=5,
          layer_widths=16,
          expansion_coefficient_dim=6,
          trend_polynomial_degree=2,
          n_epochs=50,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'france_forecasts_nbeatsx_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the forecast generation
  """
  # Initialize forecast generator
  generator = franceForecastGenerator(
      data_path='all_mulvar_data_france_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"france_forecasts_nbeatsx_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# france_forecasts_12m.csv
# france_forecasts_24m.csv

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params
-----------------------------------------------------
0 | criterion       | MSELoss          | 0     
1 | train_criterion | MSELoss          | 0     
2 | val_criterion   | MSELoss          | 0     
3 | train_metrics   | MetricCollection | 0     
4 | val_metrics     | MetricCollection | 0     
5 | stacks          | ModuleList       | 64.4 K
-----------------------------------------------------
64.1 K    Trainable params
291       Non-trainable params
64.4 K    Total params
0.258     Total esti


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to france_forecasts_nbeatsx_24m.csv

Created files:
france_forecasts_nbeatsx_12m.csv
france_forecasts_nbeatsx_24m.csv


**Forecasts**
 - **Country: Germany**
 - **Forecast Horizons:12M and 24M Forecast**

In [5]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class germanyForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the forecast generator for germany data

      Args:
          data_path: Path to the germany data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> NBEATSModel:
      """
      Create and return the N-BEATS model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured NBEATSModel
      """
      return NBEATSModel(
          input_chunk_length=3,  #forecast_horizon,
          output_chunk_length=forecast_horizon,
          num_stacks=12,
          num_blocks=2,
          num_layers=5,
          layer_widths=16,
          expansion_coefficient_dim=6,
          trend_polynomial_degree=2,
          n_epochs=50,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'germany_forecasts_nbeatsx_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the forecast generation
  """
  # Initialize forecast generator
  generator = germanyForecastGenerator(
      data_path='all_mulvar_data_germany_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"germany_forecasts_nbeatsx_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# germany_forecasts_12m.csv
# germany_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params
-----------------------------------------------------
0 | criterion       | MSELoss          | 0     
1 | train_criterion | MSELoss          | 0     
2 | val_criterion   | MSELoss          | 0     
3 | train_metrics   | MetricCollection | 0     
4 | val_metrics     | MetricCollection | 0     
5 | stacks          | ModuleList       | 64.4 K
-----------------------------------------------------
64.1 K    Trainable params
291       Non-trainable params
64.4 K    Total params
0.258     Total esti

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to germany_forecasts_nbeatsx_24m.csv

Created files:
germany_forecasts_nbeatsx_12m.csv
germany_forecasts_nbeatsx_24m.csv


**Forecasts**
 - **Country: Japan**
 - **Forecast Horizons:12M and 24M Forecast**

In [6]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class japanForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the forecast generator for japan data

      Args:
          data_path: Path to the japan data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> NBEATSModel:
      """
      Create and return the N-BEATS model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured NBEATSModel
      """
      return NBEATSModel(
          input_chunk_length=3,  #forecast_horizon,
          output_chunk_length=forecast_horizon,
          num_stacks=12,
          num_blocks=2,
          num_layers=5,
          layer_widths=16,
          expansion_coefficient_dim=6,
          trend_polynomial_degree=2,
          n_epochs=50,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'japan_forecasts_nbeatsx_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the forecast generation
  """
  # Initialize forecast generator
  generator = japanForecastGenerator(
      data_path='all_mulvar_data_japan_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"japan_forecasts_nbeatsx_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# japan_forecasts_12m.csv
# japan_forecasts_24m.csv

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params
-----------------------------------------------------
0 | criterion       | MSELoss          | 0     
1 | train_criterion | MSELoss          | 0     
2 | val_criterion   | MSELoss          | 0     
3 | train_metrics   | MetricCollection | 0     
4 | val_metrics     | MetricCollection | 0     
5 | stacks          | ModuleList       | 64.4 K
-----------------------------------------------------
64.1 K    Trainable params
291       Non-trainable params
64.4 K    Total params
0.258     Total esti


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to japan_forecasts_nbeatsx_24m.csv

Created files:
japan_forecasts_nbeatsx_12m.csv
japan_forecasts_nbeatsx_24m.csv


**Forecasts**
 - **Country: UK**
 - **Forecast Horizons:12M and 24M Forecast**

In [7]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class ukForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the forecast generator for uk data

      Args:
          data_path: Path to the uk data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> NBEATSModel:
      """
      Create and return the N-BEATS model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured NBEATSModel
      """
      return NBEATSModel(
          input_chunk_length=3,  #forecast_horizon,
          output_chunk_length=forecast_horizon,
          num_stacks=12,
          num_blocks=2,
          num_layers=5,
          layer_widths=16,
          expansion_coefficient_dim=6,
          trend_polynomial_degree=2,
          n_epochs=50,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'uk_forecasts_nbeatsx_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the forecast generation
  """
  # Initialize forecast generator
  generator = ukForecastGenerator(
      data_path='all_mulvar_data_uk_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"uk_forecasts_nbeatsx_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# uk_forecasts_12m.csv
# uk_forecasts_24m.csv


Generating 12-month forecast...
Training model for 12-month horizon...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params
-----------------------------------------------------
0 | criterion       | MSELoss          | 0     
1 | train_criterion | MSELoss          | 0     
2 | val_criterion   | MSELoss          | 0     
3 | train_metrics   | MetricCollection | 0     
4 | val_metrics     | MetricCollection | 0     
5 | stacks          | ModuleList       | 64.4 K
-----------------------------------------------------
64.1 K    Trainable params
291       Non-trainable params
64.4 K    Total params
0.258     Total esti

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to uk_forecasts_nbeatsx_24m.csv

Created files:
uk_forecasts_nbeatsx_12m.csv
uk_forecasts_nbeatsx_24m.csv


**Forecasts**
 - **Country: Italy**
 - **Forecast Horizons:12M and 24M Forecast**

In [8]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class italyForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the forecast generator for italy data

      Args:
          data_path: Path to the italy data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> NBEATSModel:
      """
      Create and return the N-BEATS model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured NBEATSModel
      """
      return NBEATSModel(
          input_chunk_length=3,  #forecast_horizon,
          output_chunk_length=forecast_horizon,
          num_stacks=12,
          num_blocks=2,
          num_layers=5,
          layer_widths=16,
          expansion_coefficient_dim=6,
          trend_polynomial_degree=2,
          n_epochs=50,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'italy_forecasts_nbeatsx_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the forecast generation
  """
  # Initialize forecast generator
  generator = italyForecastGenerator(
      data_path='all_mulvar_data_italy_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"italy_forecasts_nbeatsx_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# italy_forecasts_12m.csv
# italy_forecasts_24m.csv

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Generating 12-month forecast...
Training model for 12-month horizon...


INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params
-----------------------------------------------------
0 | criterion       | MSELoss          | 0     
1 | train_criterion | MSELoss          | 0     
2 | val_criterion   | MSELoss          | 0     
3 | train_metrics   | MetricCollection | 0     
4 | val_metrics     | MetricCollection | 0     
5 | stacks          | ModuleList       | 64.4 K
-----------------------------------------------------
64.1 K    Trainable params
291       Non-trainable params
64.4 K    Total params
0.258     Total estimated model params size (MB)


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to italy_forecasts_nbeatsx_24m.csv

Created files:
italy_forecasts_nbeatsx_12m.csv
italy_forecasts_nbeatsx_24m.csv
